# Wayfinder Eval Pipeline v3 — Multi-Lens Geometry

Runs the full eval suite on the **v3 proof network** with typed anchor geometry:

1. **Eval Retrieval** (EXP-2.1): Nav vs Dense recall with multi-lens coherence scoring
2. **Eval Spreading** (EXP-2.2): Spreading activation benefit
3. **Anchor Gap Analysis** (EXP-0.2): Category-aware gap analysis

**v3 changes:** Anchors carry `{label, category, confidence}` across 6 lens categories.
Scoring uses geometric mean across populated lenses (multi-lens coherence).
`general` → confidence 0.0, `.lake/packages` → 0.05, premise-only entities get no proof lens.

**Pre-requisite:** Upload `proof_network_v3.db` to Google Drive (built locally).

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Clone repo and install deps
!git clone https://github.com/rohanvinaik/Wayfinder.git /content/wayfinder
%cd /content/wayfinder
!pip install -q torch numpy pyyaml sentence-transformers transformers

In [ ]:
# Verify GPU available
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Runtime → Change runtime type → T4 GPU')

In [ ]:
# Mount Google Drive and copy data files
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

drive_base = '/content/drive/MyDrive/Wayfinder'  # <-- EDIT this path

# v3 DB (pre-built locally with typed anchors) + eval artifacts
file_map = {
    f'{drive_base}/proof_network_v3.db': 'data/proof_network.db',
    f'{drive_base}/nav_eval.jsonl': 'data/nav_eval.jsonl',
    f'{drive_base}/nav_train.jsonl': 'data/nav_train.jsonl',
    f'{drive_base}/NAV-002_step5000.pt': 'models/NAV-002_step5000.pt',
}

# Also try non-versioned name as fallback for the DB
if not os.path.exists(f'{drive_base}/proof_network_v3.db'):
    if os.path.exists(f'{drive_base}/proof_network.db'):
        file_map[f'{drive_base}/proof_network.db'] = 'data/proof_network.db'
        print('  Note: using proof_network.db (no v3 suffix found)')

for src, dst in file_map.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'  Copying {os.path.basename(src)} ({os.path.getsize(src)/1e6:.0f} MB)...')
            shutil.copy2(src, dst)
        else:
            print(f'  ✓ {dst} already exists')
    else:
        print(f'  ✗ {src} NOT FOUND')

In [ ]:
# Verify v3 DB has typed anchor categories
import sqlite3

conn = sqlite3.connect('data/proof_network.db')

n_entities = conn.execute('SELECT COUNT(*) FROM entities').fetchone()[0]
n_anchors = conn.execute('SELECT COUNT(*) FROM anchors').fetchone()[0]
n_traced = conn.execute("SELECT COUNT(*) FROM entities WHERE provenance='traced'").fetchone()[0]
n_premise = conn.execute("SELECT COUNT(*) FROM entities WHERE provenance='premise_only'").fetchone()[0]

print(f'Entities: {n_entities:,} ({n_traced:,} traced, {n_premise:,} premise_only)')
print(f'Anchors: {n_anchors:,}')

# Verify categories are populated (not all 'general')
cats = conn.execute(
    'SELECT category, COUNT(*) FROM anchors GROUP BY category ORDER BY COUNT(*) DESC'
).fetchall()
print('\nAnchor categories:')
for cat, count in cats:
    print(f'  {cat}: {count:,}')

if len(cats) <= 1:
    print('\n✗ WARNING: Only one anchor category — this may be the v2 DB, not v3!')
    print('  Upload proof_network_v3.db (built locally with multi-lens anchors)')
else:
    print(f'\n✓ v3 DB confirmed: {len(cats)} anchor categories')

conn.close()

---
## Experiment 1: Eval Retrieval (EXP-2.1)

Compares navigational retrieval vs dense cosine similarity baseline.
Now uses **multi-lens coherence scoring**: per-category anchor overlap with
geometric mean across populated lenses.

**Gate:** nav recall@16 ≥ 80% of dense recall@16

In [ ]:
import os

os.chdir('/content/wayfinder')
os.environ['PYTHONPATH'] = '/content/wayfinder'

!PYTHONPATH=/content/wayfinder python scripts/eval_retrieval.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_retrieval_v3_gpu.json

In [ ]:
import json
import os

result_path = 'runs/eval_retrieval_v3_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_retrieval output above for errors.')
else:
    with open(result_path) as f:
        report = json.load(f)

    print('=== Universe Coverage ===')
    cov = report['universe_coverage']
    print(f"  Mean: {cov['mean']:.1%}")
    print(f"  Fully covered: {cov['fully_covered']}/{report['samples']}")
    print(f"  Zero covered: {cov['zero_covered']}/{report['samples']}")

    print('\n=== Raw Retrieval ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_retrieval'][f'recall@{k}']
        dr = report['dense_retrieval'][f'recall@{k}']
        print(f"  recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print('\n=== Conditional Retrieval (reachable premises only) ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_conditional_retrieval'][f'cond_recall@{k}']
        dr = report['dense_conditional_retrieval'][f'cond_recall@{k}']
        print(f"  cond_recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print('\n=== Timing ===')
    t = report['timing']
    print(f"  Nav: {t['nav_avg_ms']:.1f}ms  Dense: {t['dense_avg_ms']:.1f}ms  Speedup: {t['speedup']:.1f}x")

---
## Experiment 2: Eval Spreading (EXP-2.2)

Tests whether spreading activation from proof history improves premise retrieval.

**Gate:** ≥5% recall@16 improvement at proof step 3+

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/eval_spreading.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_spreading_v3_gpu.json

In [ ]:
import json
import os

result_path = 'runs/eval_spreading_v3_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_spreading output above for errors.')
else:
    with open(result_path) as f:
        spread_report = json.load(f)
    print(json.dumps(spread_report, indent=2))

---
## Experiment 3: Anchor Gap Analysis (EXP-0.2)

Category-aware gap analysis on the v3 DB with typed anchors.
Reports per-lens gap distribution and excludes trivial anchors
(`general`, `.lake/packages` paths) from gap surfacing.

**Gate:** recall@16 ≥ 70% on perfect queries

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/anchor_gap_analysis.py \
    --db data/proof_network.db \
    --samples 500 \
    --output runs/anchor_gap_v3.jsonl

---
## Download Results

In [ ]:
import os
import shutil

result_files = [
    'runs/eval_retrieval_v3_gpu.json',
    'runs/eval_spreading_v3_gpu.json',
    'runs/anchor_gap_v3.jsonl',
]

os.makedirs('/content/results', exist_ok=True)
for f in result_files:
    if os.path.exists(f):
        shutil.copy(f, f'/content/results/{os.path.basename(f)}')
        print(f'  ✓ {f}')
    else:
        print(f'  ✗ {f} (not found)')

shutil.make_archive('/content/wayfinder_eval_results_v3', 'zip', '/content/results')
print('\nDownload: /content/wayfinder_eval_results_v3.zip')

try:
    from google.colab import files
    files.download('/content/wayfinder_eval_results_v3.zip')
except ImportError:
    print('Not in Colab — download manually from the file browser')